In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import MinMaxScaler
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt


In [ ]:
df = pd.read_csv("../data/time_series.csv")
values = df["value"].values.reshape(-1,1)


In [ ]:
scaler = MinMaxScaler()
values = scaler.fit_transform(values)


In [ ]:
class SeqDataset(Dataset):
    def __init__(self, data, seq_len=30):
        self.data = data
        self.seq_len = seq_len

    def __len__(self):
        return len(self.data) - self.seq_len

    def __getitem__(self, idx):
        seq = self.data[idx:idx+self.seq_len]
        label = self.data[idx+self.seq_len]
        return torch.FloatTensor(seq), torch.FloatTensor(label)


In [ ]:
seq_len = 30
dataset = SeqDataset(values, seq_len)
loader = DataLoader(dataset, batch_size=32, shuffle=True)


In [ ]:
class LSTMModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(1, 64, batch_first=True)
        self.fc = nn.Linear(64, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.fc(out[:, -1])
        return out


In [ ]:
model = LSTMModel()
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(30):
    for seq, label in loader:
        optimizer.zero_grad()
        output = model(seq)
        loss = criterion(output, label)
        loss.backward()
        optimizer.step()
    print("Epoch:", epoch, "Loss:", loss.item())


In [ ]:
test_input = values[-seq_len:].reshape(1, seq_len, 1)
predictions = []

for _ in range(20):
    with torch.no_grad():
        pred = model(torch.FloatTensor(test_input))
        predictions.append(pred.item())
        test_input = np.append(test_input[:,1:,:], [[pred]], axis=1)

pred_values = scaler.inverse_transform(np.array(predictions).reshape(-1,1))
